In [1]:
import os
import pandas as pd

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")

In [2]:
import dspy
lm = dspy.LM('openai/gpt-5.1', temperature=1.0, max_tokens=16000, api_key=OPENAI_API_KEY)
dspy.configure(lm=lm)

# Dataset

In [3]:
import dspy
import pandas as pd

def init_dataset_from_df(df, split_ratio=(0.8, 0.1, 0.1), seed=42):
    assert abs(sum(split_ratio) - 1.0) < 1e-6, "split_ratio must sum to 1.0"
    assert {'text', 'polarization'}.issubset(df.columns), "DataFrame must have 'text' and 'polarization' columns"

    # Shuffle deterministically
    df = df.sample(frac=1, random_state=seed).reset_index(drop=True)
    total = len(df)

    # Split boundaries
    train_end = int(split_ratio[0] * total)
    val_end = train_end + int(split_ratio[1] * total)

    train_df = df.iloc[:train_end]
    val_df = df.iloc[train_end:val_end]
    test_df = df.iloc[val_end:]

    # Convert to DSPy examples
    def to_dspy_list(sub_df):
        return [
            dspy.Example({
                "statement": row["text"],
                "answer": row["polarization"],
            }).with_inputs("statement")
            for _, row in sub_df.iterrows()
        ]

    train_set = to_dspy_list(train_df)
    val_set = to_dspy_list(val_df)
    test_set = to_dspy_list(test_df)

    return train_set, val_set, test_set


# GEPA

In [4]:
from data_utils import split_dataframe

In [9]:
filename = "../data/dev_phase/subtask1/train/eng.csv"
english_train_df = pd.read_csv(filename)
english_train_df = english_train_df.sample(n=120, random_state=42).reset_index(drop=True)

train_df, val_df, test_df = init_dataset_from_df(english_train_df)

In [10]:
len(train_df), len(val_df), len(test_df)

(96, 12, 12)

In [13]:
# Get the first example
example = train_df[0]

# Access its fields
print(example)                # Shows full Example object

Example({'statement': 'When you are a victim of a crime they blame the system. When they are a victim of a crime they blame YOU! pelosiattack DefundThePolice', 'answer': 0}) (input_keys={'statement'})


In [15]:
def metric_with_feedback(example, prediction, trace=None, pred_name=None, pred_trace=None):
    statements = example["statement"]
    if not isinstance(statements, list):
        statements = [statements]

    correct_answers = example["answer"]
    if not isinstance(correct_answers, list):
        correct_answers = [str(correct_answers)]  # convert int to str

    llm_answers = prediction.answer
    if not isinstance(llm_answers, list):
        llm_answers = [str(llm_answers)]

    if len(statements) != len(correct_answers) or len(statements) != len(llm_answers):
        return dspy.Prediction(
            score=0,
            feedback="Invalid input: lengths of statements, answers, and predictions do not match."
        )

    total = len(correct_answers)
    correct_count = 0
    feedbacks = []

    for i, (correct, statement, pred) in enumerate(
        zip(correct_answers, statements, llm_answers)
    ):
        if pred not in {'0', '1'}:
            feedback_text = (
                f"The model must output only '1' or '0' to indicate whether the statement is polarizing. "
                f"You responded with '{pred}', which is invalid. "
                f"The correct label for this statement is '{correct}'."
            )
        elif pred == correct:
            correct_count += 1
            feedback_text = f"Correct — the statement is labeled '{correct}'."
        else:
            feedback_text = (
                f"[Item {i}] Incorrect — the correct label is '{correct}', but you predicted '{pred}'. "
            )
        if statement:
            feedback_text += f"\n\nStatement:\n{statement}\n\n"

        feedbacks.append(feedback_text)

    score = correct_count / total if total > 0 else 0

    return dspy.Prediction(score=score, feedback="\n\n".join(feedbacks))

In [16]:
class GenerateResponse(dspy.Signature):
    statement = dspy.InputField()
    answer = dspy.OutputField()

program = dspy.ChainOfThought(GenerateResponse)

In [17]:
example = {
    "statement": ["Some text", "Some other text"],
    "answer": ["1", "0"],  # correct label
}

prediction = dspy.Prediction(
    answer=["1", "0"]       # model output
)

result = metric_with_feedback(example, prediction)

print(result.score)
print(result.feedback)

1.0
Correct — the statement is labeled '1'.

Statement:
Some text



Correct — the statement is labeled '0'.

Statement:
Some other text




In [18]:
from dspy import GEPA

optimizer = GEPA(
    metric=metric_with_feedback,
    auto="light",
    num_threads=32,
    track_stats=True,
    reflection_minibatch_size=3,
    reflection_lm=dspy.LM('openai/gpt-5.1', temperature=1.0, max_tokens=16000, api_key=OPENAI_API_KEY)

)

optimized_program = optimizer.compile(
    program,
    trainset=train_df,
    valset=val_df,
)

2025/12/24 00:45:48 INFO dspy.teleprompt.gepa.gepa: Running GEPA for approx 428 metric calls of the program. This amounts to 3.96 full evals on the train+val set.
2025/12/24 00:45:48 INFO dspy.teleprompt.gepa.gepa: Using 12 examples for tracking Pareto scores. You can consider using a smaller sample of the valset to allow GEPA to explore more diverse solutions within the same budget.
GEPA Optimization:   0%|          | 0/428 [00:00<?, ?rollouts/s]2025/12/24 00:45:59 INFO dspy.evaluate.evaluate: Average Metric: 0.0 / 12 (0.0%)
2025/12/24 00:45:59 INFO dspy.teleprompt.gepa.gepa: Iteration 0: Base program full valset score: 0.0
GEPA Optimization:   3%|▎         | 12/428 [00:11<06:27,  1.07rollouts/s]2025/12/24 00:45:59 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Selected program 0 score: 0.0


Average Metric: 0.00 / 3 (0.0%): 100%|██████████| 3/3 [00:04<00:00,  1.65s/it]

2025/12/24 00:46:04 INFO dspy.evaluate.evaluate: Average Metric: 0.0 / 3 (0.0%)


2025/12/24 00:46:12 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Proposed new text for predict: You are given a single input field:

- `statement`: a short piece of user-generated text, often about news, media, politics, current events, or contentious topics.

Your task is to output a single field:

- `answer`: a *binary label* indicating whether the `statement` is **polarizing**.

A statement is considered **polarizing (label = '1')** if, in typical public or political discourse, it is likely to:
- Express, imply, or clearly align with a **partisan, ideological, or strongly value‑laden stance**, or
- Use **derogatory, inflammatory, or strongly negative** language toward a political actor, group, or side in a contentious issue, or
- Make a *charged* or *value‑laden* claim about a political or identity group, a nation, a conflict, the media, or institutions (e.g., accusations of “genocide,” “fake news,” “traitors,” etc.), or
- Be the kind of statement that would predictably provoke **st

Average Metric: 2.00 / 3 (66.7%): 100%|██████████| 3/3 [00:01<00:00,  2.02it/s] 

...

Average Metric: 3.00 / 3 (100.0%): 100%|██████████| 3/3 [00:01<00:00,  1.89it/s]

2025/12/24 00:56:46 INFO dspy.evaluate.evaluate: Average Metric: 3.0 / 3 (100.0%)
2025/12/24 00:56:46 INFO dspy.teleprompt.gepa.gepa: Iteration 37: All subsample scores perfect. Skipping.
2025/12/24 00:56:46 INFO dspy.teleprompt.gepa.gepa: Iteration 37: Reflective mutation did not propose a new candidate
GEPA Optimization:  99%|█████████▉| 423/428 [10:57<00:08,  1.75s/rollouts]2025/12/24 00:56:46 INFO dspy.teleprompt.gepa.gepa: Iteration 38: Selected program 20 score: 1.0



Average Metric: 1.00 / 3 (33.3%): 100%|██████████| 3/3 [00:01<00:00,  2.36it/s] 

2025/12/24 00:56:47 INFO dspy.evaluate.evaluate: Average Metric: 1.0 / 3 (33.3%)


2025/12/24 00:57:20 INFO dspy.teleprompt.gepa.gepa: Iteration 38: Proposed new text for predict: You are an AI assistant that performs **binary classification** on short text statements.

Your sole job is to decide whether each input `statement` is **polarizing** or **non‑polarizing** under the specific scheme below, and then output a single character: `1` or `0`.

You must follow these instructions exactly.

--------------------------------
TASK OVERVIEW
--------------------------------

Input:

- `statement`: a short piece of user‑generated text, often about news, media, politics, current events, or other contentious topics.

Output:

- A single character `answer`:
  - `1` if the `statement` is **polarizing**
  - `0` if the `statement` is **non‑polarizing**

You must obey the **OUTPUT FORMAT** rules at the end.

--------------------------------
CORE CLASSIFICATION TASK
--------------------------------

Classify each `statement` as:

- **`1` (polarizing)** if it clearly takes or impli

In [30]:
print(optimized_program.detailed_results.highest_score_achieved_per_val_task)

[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
